In [ ]:
import os
import matplotlib.pyplot as plt
import networkx as nx
import numpy as np
import pandas as pd
import scipy.sparse as sp
import seaborn as sns
from sklearn.manifold import TSNE
from sklearn.metrics import confusion_matrix, classification_report

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim



## Dataset

Before diving into the models, we need to toy around with the cora dataset. The functions presented in this notebook come directly from this [website](https://graphsandnetworks.com/the-cora-dataset/).

In [5]:
data_dir ="cora\cora"

In [6]:
edgelist = pd.read_csv(os.path.join(data_dir, "cora.cites"), sep='\t', header=None, names=["target", "source"])
edgelist["label"] = "cites"

In [9]:
edgelist.sample(frac=1).head(5)


,target,source,label
1877,12576,1105221,cites
1557,8224,6639,cites
4778,289885,20924,cites
5103,594483,1129040,cites
3676,77758,25772,cites


In [10]:
Gnx = nx.from_pandas_edgelist(edgelist, edge_attr="label")
nx.set_node_attributes(Gnx, "paper", "label")

In [11]:
Gnx.nodes[1103985]

{'label': 'paper'}

In [13]:
feature_names = ["w_{}".format(ii) for ii in range(1433)]
column_names =  feature_names + ["subject"]
node_data = pd.read_csv(os.path.join(data_dir, "cora.content"), sep='\t', header=None, names=column_names)

In [14]:
node_data.head(5)


,w_0,w_1,w_2,w_3,w_4,w_5,w_6,w_7,w_8,w_9,...,w_1424,w_1425,w_1426,w_1427,w_1428,w_1429,w_1430,w_1431,w_1432,subject
31336,0,0,0,0,0,0,0,0,0,0,...,0,0,1,0,0,0,0,0,0,Neural_Networks
1061127,0,0,0,0,0,0,0,0,0,0,...,0,1,0,0,0,0,0,0,0,Rule_Learning
1106406,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,Reinforcement_Learning
13195,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,Reinforcement_Learning
37879,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,Probabilistic_Methods


In [15]:
set(node_data["subject"])

{'Case_Based',
 'Genetic_Algorithms',
 'Neural_Networks',
 'Probabilistic_Methods',
 'Reinforcement_Learning',
 'Rule_Learning',
 'Theory'}

In [ ]:
# We extract the words features (1433 dimensions) from the node data
raw_features = node_data.iloc[:, :-1].values

# Conversion to PyTorch tensor
features = torch.FloatTensor(raw_features)

print(f"Features : {features.shape}") 

Features : torch.Size([2708, 1433])


## Models

We will implement three models to start with:
- Standard MLP with two fully connected layers. This network will highlight the benefits of adopting the "graph" structure in the GNNs.
- GCN with two layers. This model will follow the paper from Kipf.
- GAT with two layers. This model will follow the paper from Veličković.

In [ ]:
class MLP(nn.Module):
    def __init__(self, n_features, n_hidden, n_classes, dropout):
        super(MLP, self).__init__()
        # Two Linear layers with ReLU and Dropout in between
        self.fc1 = nn.Linear(n_features, n_hidden)
        self.fc2 = nn.Linear(n_hidden, n_classes)
        self.dropout = dropout

    def forward(self, x):
        # First layer + ReLU + Dropout
        x = F.relu(self.fc1(x))
        x = F.dropout(x, self.dropout, training=self.training)
        
        # Second layer (logits)
        x = self.fc2(x)
        
        return F.log_softmax(x, dim=1)

In [ ]:
class GCNLayer(nn.Module):
    def __init__(self, in_features, out_features):
        super(GCNLayer, self).__init__()
        # Linear Transform
        self.projection = nn.Linear(in_features, out_features, bias=False)

    def forward(self, x, adj_norm):
        # Feature transform : X * W
        x = self.projection(x)
        
        # Propagation : A_hat * (X * W)
        # We use torch.mm for matrix multiplication for efficiency
        x = torch.mm(adj_norm, x)
        
        return x

class GCN(nn.Module):
    def __init__(self, n_features, n_hidden, n_classes, dropout):
        super(GCN, self).__init__()
        self.layer1 = GCNLayer(n_features, n_hidden)
        self.layer2 = GCNLayer(n_hidden, n_classes)
        self.dropout = dropout

    def forward(self, x, adj_norm):
        # First Layer + Activation + Dropout
        x = F.relu(self.layer1(x, adj_norm))
        x = F.dropout(x, self.dropout, training=self.training)
        
        # Second Layer (output logits for classification)
        x = self.layer2(x, adj_norm)
        
        return F.log_softmax(x, dim=1)

For the GCN, we need to normalize the adjacency matrix.

In [ ]:
def normalize_adjacency(adj):
    """
    Returns the normalized adjacency matrix A_hat = D^-1/2 * (A + I) * D^-1/2
    """
    # Add the self-loops (A + I) with scipy sparse for efficiency
    adj = sp.coo_matrix(adj)
    adj_tilde = adj + sp.eye(adj.shape[0])
    
    # Compute the degree matrix D_tilde (sum of rows of A_tilde)
    rowsum = np.array(adj_tilde.sum(1))
    
    # D_tilde^-1/2
    d_inv_sqrt = np.power(rowsum, -0.5).flatten()
    d_inv_sqrt[np.isinf(d_inv_sqrt)] = 0. # Division by zero handling
    d_mat_inv_sqrt = sp.diags(d_inv_sqrt)
    
    # D^-1/2 * A_tilde * D^-1/2
    adj_norm = d_mat_inv_sqrt.dot(adj_tilde).dot(d_mat_inv_sqrt).tocoo()
    
    return adj_norm

In [20]:
def sparse_mx_to_torch_sparse_tensor(sparse_mx):
    """Convert a scipy sparse matrix to a torch sparse tensor."""
    sparse_mx = sparse_mx.tocoo().astype(np.float32)
    indices = torch.from_numpy(
        np.vstack((sparse_mx.row, sparse_mx.col)).astype(np.int64)
    )
    values = torch.from_numpy(sparse_mx.data)
    shape = torch.Size(sparse_mx.shape)
    return torch.sparse_coo_tensor(indices, values, shape)

In [ ]:
# Labels encoding: we convert the string labels to integer indices for PyTorch
labels_raw = node_data["subject"].values
unique_labels = np.unique(labels_raw)
label_map = {l: i for i, l in enumerate(unique_labels)}
labels = torch.LongTensor([label_map[l] for l in labels_raw])

In [ ]:
# Mapping ID -> Index 
node_ids = node_data.index.values
node_map = {j: i for i, j in enumerate(node_ids)}

# Load the edges
edges_unordered = pd.read_csv(os.path.join(data_dir, "cora.cites"), sep='\t', header=None).values

# Map the edge IDs to our 0,...,2707 indices
# We ignore edges that point to nodes absent from cora.content
edges = np.array(list(map(node_map.get, edges_unordered.flatten())),
                 dtype=np.int32).reshape(edges_unordered.shape)

# Build the adjacency matrix A (sparse) from the edges
adj = sp.coo_matrix((np.ones(edges.shape[0]), (edges[:, 0], edges[:, 1])),
                    shape=(labels.shape[0], labels.shape[0]),
                    dtype=np.float32)

# We make it symmetric (if A propagates to B, then B propagates to A)
adj = adj + adj.T.multiply(adj.T > adj) - adj.multiply(adj.T > adj)

# Now we apply the normalization (using the normalize_adjacency function defined above) to match the GCN 
adj_norm = normalize_adjacency(adj)

# Convert to Torch Sparse Tensor
adj_tensor = sparse_mx_to_torch_sparse_tensor(adj_norm)

In [32]:
adj_tensor.shape

torch.Size([2708, 2708])

In [ ]:
class GATLayer(nn.Module):
    def __init__(self, in_features, out_features, dropout, alpha, concat=True):
        super(GATLayer, self).__init__()
        self.in_features = in_features
        self.out_features = out_features
        self.dropout = dropout
        self.alpha = alpha # LeakyRelu slope
        self.concat = concat

        # Shared Linear Transformation (W)
        self.W = nn.Linear(in_features, out_features, bias=False)
        
        # Attention mechanism (a)
        self.a = nn.Parameter(torch.empty(size=(2 * out_features, 1)))
        nn.init.xavier_uniform_(self.a.data, gain=1.414)

        self.leakyrelu = nn.LeakyReLU(self.alpha)

    def forward(self, h, adj):
        # h: nodes features [N, in_features]
        # adj: dense adjacency matrix [N, N] 
        
        Wh = self.W(h) # [N, out_features]
        N = Wh.size()[0]

        # Self-attention
        # We concatenate the features of each pair of nodes (N x N)
        a_input = torch.cat([Wh.repeat(1, N).view(N * N, -1), Wh.repeat(N, 1)], dim=1).view(N, -1, 2 * self.out_features)
        # e is the matrix of raw attention scores
        e = self.leakyrelu(torch.matmul(a_input, self.a).squeeze(2)) # [N, N]

        # Masking : we only keep scores where there is an edge (A_ij > 0)
        zero_vec = -9e15 * torch.ones_like(e)
        attention = torch.where(adj > 0, e, zero_vec)
        
        # We normalize with softmax across the neighbors
        attention = F.softmax(attention, dim=1)
        attention = F.dropout(attention, self.dropout, training=self.training)

        # Final output : weighted sum of neighbors' features
        h_prime = torch.matmul(attention, Wh)

        if self.concat:
            return F.elu(h_prime)
        else:
            return h_prime

class GAT(nn.Module):
    def __init__(self, nfeat, nhid, nclass, dropout, alpha, nheads):
        super(GAT, self).__init__()
        self.dropout = dropout

        # Multi-head attention in the first layer
        self.attentions = nn.ModuleList([
            GATLayer(nfeat, nhid, dropout=dropout, alpha=alpha, concat=True) 
            for _ in range(nheads)
        ])

        # Output layer (single head, without concatenation)
        self.out_att = GATLayer(nhid * nheads, nclass, dropout=dropout, alpha=alpha, concat=False)

    def forward(self, x, adj):
        x = F.dropout(x, self.dropout, training=self.training)
        
        # We concatenate the outputs of the multi-head attention
        x = torch.cat([att(x, adj) for att in self.attentions], dim=1)
        
        x = F.dropout(x, self.dropout, training=self.training)
        x = F.log_softmax(self.out_att(x, adj), dim=1)
        
        return x
    
# We convert the adjacency matrix to dense format for GAT, since it uses dense operations
# We use the adjacency matrix with self-loops (A + I) but WITHOUT the normalization D^-1/2
adj_tilde = adj + sp.eye(adj.shape[0])
adj_dense = torch.FloatTensor(adj_tilde.todense())

## Training

Now that we created our models, we are ready to train them using masks.

In [ ]:
# Masks for the splits train/val/test

N = 2708  # Total number of nodes in Cora

# Initialize the masks to False
train_mask = torch.zeros(N, dtype=torch.bool)
val_mask = torch.zeros(N, dtype=torch.bool)
test_mask = torch.zeros(N, dtype=torch.bool)

# We train on a reduced number of nodes to speed up training and avoid overfitting
train_mask[:140] = True
val_mask[140:640] = True
test_mask[1708:2708] = True

In [85]:
def accuracy(output, labels):
    preds = output.max(1)[1].type_as(labels)
    correct = preds.eq(labels).double()
    correct = correct.sum()
    return correct / len(labels)

def train(epochs, model, optimizer, criterion, adj_tensor, is_mlp=True):

    history = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []}
    for epoch in range(epochs):
        model.train()
        optimizer.zero_grad()
        
        if is_mlp:
            output = model(features)
        else:
            output = model(features, adj_tensor)
        loss_train = criterion(output[train_mask], labels[train_mask])
        acc_train = accuracy(output[train_mask], labels[train_mask])
        
        loss_train.backward()
        optimizer.step()

        model.eval()
        with torch.no_grad():
            if is_mlp:
                output = model(features)
            else:
                output = model(features, adj_tensor)
            loss_val = criterion(output[val_mask], labels[val_mask])
            acc_val = accuracy(output[val_mask], labels[val_mask])
        
        history['train_loss'].append(loss_train.item())
        history['train_acc'].append(acc_train.item())
        history['val_loss'].append(loss_val.item())
        history['val_acc'].append(acc_val.item())

        if epoch % 10 == 0:
            print(f'Epoch: {epoch:03d}, Loss Train: {loss_train.item():.4f}, '
                  f'Acc Train: {acc_train:.4f}, Acc Val: {acc_val:.4f}')

    return history

In [86]:
# Initialize the MLP model, optimizer and loss function
model_mlp = MLP(n_features=1433, n_hidden=16, n_classes=7, dropout=0.5)
optimizer_mlp = optim.Adam(model_mlp.parameters(), lr=0.01, weight_decay=5e-4)
criterion_mlp = torch.nn.NLLLoss()

history_mlp = train(200, model_mlp, optimizer_mlp, criterion_mlp, adj_tensor, is_mlp=True)

Epoch: 000, Loss Train: 1.9341, Acc Train: 0.2071, Acc Val: 0.1360
Epoch: 010, Loss Train: 1.0650, Acc Train: 0.7357, Acc Val: 0.4980
Epoch: 020, Loss Train: 0.4653, Acc Train: 0.8357, Acc Val: 0.5500
Epoch: 030, Loss Train: 0.3846, Acc Train: 0.8643, Acc Val: 0.5720
Epoch: 040, Loss Train: 0.2681, Acc Train: 0.8857, Acc Val: 0.5640
Epoch: 050, Loss Train: 0.2685, Acc Train: 0.8929, Acc Val: 0.5660
Epoch: 060, Loss Train: 0.2606, Acc Train: 0.9214, Acc Val: 0.5660
Epoch: 070, Loss Train: 0.1505, Acc Train: 0.9429, Acc Val: 0.5700
Epoch: 080, Loss Train: 0.2025, Acc Train: 0.9286, Acc Val: 0.5680
Epoch: 090, Loss Train: 0.2270, Acc Train: 0.9071, Acc Val: 0.5660
Epoch: 100, Loss Train: 0.1682, Acc Train: 0.9571, Acc Val: 0.5720
Epoch: 110, Loss Train: 0.1852, Acc Train: 0.9357, Acc Val: 0.5780
Epoch: 120, Loss Train: 0.1890, Acc Train: 0.9286, Acc Val: 0.5760
Epoch: 130, Loss Train: 0.1501, Acc Train: 0.9357, Acc Val: 0.5800
Epoch: 140, Loss Train: 0.1390, Acc Train: 0.9643, Acc Val: 0.

In [87]:
# Initialize the GCN model, optimizer and loss function
model_gcn = GCN(n_features=1433, n_hidden=16, n_classes=7, dropout=0.7)
optimizer_gcn = optim.Adam(model_gcn.parameters(), lr=0.01, weight_decay=5e-4)
criterion_gcn = torch.nn.NLLLoss()

history_gcn = train(200, model_gcn, optimizer_gcn, criterion_gcn, adj_tensor = adj_tensor, is_mlp = False)

Epoch: 000, Loss Train: 1.9444, Acc Train: 0.1786, Acc Val: 0.5320
Epoch: 010, Loss Train: 1.0266, Acc Train: 0.7714, Acc Val: 0.6380
Epoch: 020, Loss Train: 0.5652, Acc Train: 0.8714, Acc Val: 0.8300
Epoch: 030, Loss Train: 0.2644, Acc Train: 0.9286, Acc Val: 0.8260
Epoch: 040, Loss Train: 0.1752, Acc Train: 0.9571, Acc Val: 0.8240
Epoch: 050, Loss Train: 0.2000, Acc Train: 0.9571, Acc Val: 0.8280
Epoch: 060, Loss Train: 0.1643, Acc Train: 0.9643, Acc Val: 0.8260
Epoch: 070, Loss Train: 0.1357, Acc Train: 0.9714, Acc Val: 0.8220
Epoch: 080, Loss Train: 0.1623, Acc Train: 0.9857, Acc Val: 0.8280
Epoch: 090, Loss Train: 0.1395, Acc Train: 0.9786, Acc Val: 0.8180
Epoch: 100, Loss Train: 0.1492, Acc Train: 0.9571, Acc Val: 0.8260
Epoch: 110, Loss Train: 0.1058, Acc Train: 0.9857, Acc Val: 0.8260
Epoch: 120, Loss Train: 0.1228, Acc Train: 0.9643, Acc Val: 0.8240
Epoch: 130, Loss Train: 0.1169, Acc Train: 0.9714, Acc Val: 0.8240
Epoch: 140, Loss Train: 0.1528, Acc Train: 0.9643, Acc Val: 0.

In [ ]:
n_heads = 4
n_hidden = 4  
dropout = 0.6
lr = 0.005
weight_decay = 5e-4

# Initialisation du modèle GAT
model_gat = GAT(nfeat=features.shape[1], nhid=n_hidden, nclass=int(labels.max()) + 1, dropout=dropout, alpha=0.2, nheads=n_heads)

optimizer_gat = optim.Adam(model_gat.parameters(), lr=lr, weight_decay=weight_decay)
criterion = torch.nn.NLLLoss()

history_gat = train(200, model_gat, optimizer_gat, criterion, adj_tensor=adj_dense, is_mlp=False)

## Metrics

In [ ]:
def plot_metrics(train_accs, val_accs, train_losses, val_losses):
    fig, ax = plt.subplots(1, 2, figsize=(15, 5))
    
    # Accuracy Graph
    ax[0].plot(train_accs, label='Train Acc')
    ax[0].plot(val_accs, label='Val Acc')
    ax[0].set_title('Accuracy au cours des époques')
    ax[0].legend()
    
    # Loss Graph
    ax[1].plot(train_losses, label='Train Loss')
    ax[1].plot(val_losses, label='Val Loss')
    ax[1].set_title('Perte (Loss) au cours des époques')
    ax[1].legend()
    
    plt.show()


def show_final_report(model, features, adj_tensor, labels, mask, is_mlp=True):
    model.eval()
    with torch.no_grad():
        if is_mlp:
            logits = model(features)
        else:
            logits = model(features, adj_tensor)
        preds = logits.max(1)[1]
    
    # We compute the classification report and confusion matrix only on the test set (mask)
    y_true = labels[mask].cpu().numpy()
    y_pred = preds[mask].cpu().numpy()
    
    print("Classification Report on Test Set:")
    print(classification_report(y_true, y_pred, target_names=list(label_map.keys())))
    
    # Heatmap of the confusion matrix
    cm = confusion_matrix(y_true, y_pred)
    plt.figure(figsize=(10, 8))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
                xticklabels=label_map.keys(), yticklabels=label_map.keys())
    plt.xlabel('Predicted Label')
    plt.ylabel('True Label')
    plt.title('Confusion Matrix (Test Set)')
    plt.show()

# show_final_report(model_gcn, features, adj_tensor, labels, test_mask, is_mlp=False)

In [ ]:
def visualize_mlp_embeddings(model, features, labels):
    model.eval()
    with torch.no_grad():
        x = model.fc1(features)
        embeddings = F.relu(x)

    tsne = TSNE(n_components=2, perplexity=30, random_state=42)
    out = tsne.fit_transform(embeddings.cpu().numpy())
    
    plt.figure(figsize=(10, 8))
    scatter = plt.scatter(out[:, 0], out[:, 1], c=labels.cpu(), cmap='rainbow', s=25, alpha=0.8)
    plt.colorbar(scatter)
    plt.title("t-SNE visualization of MLP's embeddings")
    plt.show()

def visualize_gcn_embeddings(model, features, adj_tensor, labels):
    model.eval()
    with torch.no_grad():
        embeddings = F.relu(model.layer1(features, adj_tensor))
        
    tsne = TSNE(n_components=2, perplexity=30, random_state=42)
    
    out = tsne.fit_transform(embeddings.cpu().numpy())
    
    plt.figure(figsize=(10, 8))
    scatter = plt.scatter(out[:, 0], out[:, 1], c=labels.cpu(), cmap='rainbow', s=20)
    plt.colorbar(scatter)
    plt.title("t-SNE visualization of GCN's embeddings")
    plt.show()

def visualize_gat_embeddings(model, features, adj_dense, labels):
    model.eval()
    with torch.no_grad():
        # We concatenate the outputs of the multi-head attention (before the output layer)
        x = torch.cat([att(features, adj_dense) for att in model.attentions], dim=1)
        # ELU is used in the GAT paper after the multi-head attention layer
        embeddings = F.elu(x)
        
    tsne = TSNE(n_components=2, perplexity=30, random_state=42)
    out = tsne.fit_transform(embeddings.cpu().numpy())
    
    plt.figure(figsize=(10, 8))
    scatter = plt.scatter(out[:, 0], out[:, 1], c=labels.cpu(), cmap='rainbow', s=25, alpha=0.8)
    plt.colorbar(scatter)
    plt.title("t-SNE visualization of GAT's embeddings")
    plt.show()

# visualize_mlp_embeddings(model_mlp, features, labels)
# visualize_gcn_embeddings(model_gcn, features, adj_tensor, labels)
# visualize_gat_embeddings(model_gat, features, adj_dense, labels)